In [1]:
# FACE PIPELINE - ONNX EXPORT NOTEBOOK
# Google Colab T4 GPU
#
# Modelos:
#   1. SCRFD-10G-KPS       -> Face Detection        (InsightFace)
#   2. MiniFASNetV2        -> Liveness Anti-Spoofing (yakhyo/face-anti-spoofing)
#   3. ArcFace buffalo_sc  -> Face Recognition 512D  (InsightFace)
#   4. EmotiEffLib         -> Emotion Recognition    (sb-ai-lab/EmotiEffLib)
#
# Execute Code Cell by Cell. Runtime: T4 GPU .

In [ ]:
import subprocess
import sys
import os

In [ ]:
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_BASE = '/content/drive/MyDrive/face_pipeline_models'
DIRS = {
    'detection'   : f'{DRIVE_BASE}/01_face_detection',
    'antispoofing': f'{DRIVE_BASE}/02_liveness_antispoofing',
    'recognition' : f'{DRIVE_BASE}/03_face_recognition',
    'emotion'     : f'{DRIVE_BASE}/04_emotion_recognition',
    'logs'        : f'{DRIVE_BASE}/export_logs',
}

In [ ]:
for path in DIRS.values():
    os.makedirs(path, exist_ok=True)
    print(f'[OK] {path}')

In [ ]:
#Install Dependencies 
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'onnx==1.15.0',
    'onnxruntime-gpu==1.17.1',
    'onnxsim==0.4.35',
    'insightface==0.7.3',
    'opencv-python-headless',
    'Pillow',
    'numpy',
    'requests',
    'tqdm',
], check=True)

In [ ]:
# MODELO 1: SCRFD Face Detection ONNX Format
#Insight Face
# SCRFD_10G_KPS: 10 GFLOPs, x 5 keypoints (KPS) by face/
# Keypoints are critical to face alignment before ArcFace

In [ ]:
import urllib.request
import hashlib

In [ ]:
def download_file(url: str, dest_path: str, desc: str = '') -> None:
    print(f'Downloading {desc} ...')
    urllib.request.urlretrieve(url, dest_path)
    size_mb = os.path.getsize(dest_path) / (1024 * 1024)
    print(f'[OK] {desc} -> {dest_path} ({size_mb:.1f} MB)'

In [ ]:
# Searching on its Repo
SCRFD_URL  = (
    'https://github.com/deepinsight/insightface/releases/download/'
    'v0.7/buffalo_sc.zip'
)
SCRFD_ZIP  = '/content/buffalo_sc.zip'
SCRFD_DIR  = '/content/buffalo_sc_raw'


In [ ]:
download_file(SCRFD_URL, SCRFD_ZIP, 'buffalo_sc (SCRFD + ArcFace bundle)')

In [ ]:
import zipfile
with zipfile.ZipFile(SCRFD_ZIP, 'r') as z:
    z.extractall(SCRFD_DIR)
    print('Archivos extraidos:')
    for name in z.namelist():
        print(f'  {name}')

In [ ]:
import shutil
import glob

In [ ]:
det_files = glob.glob(f'{SCRFD_DIR}/**/det_*.onnx', recursive=True)
rec_files = glob.glob(f'{SCRFD_DIR}/**/w600k_*.onnx', recursive=True)


In [ ]:
print(f'\nModelos de deteccion encontrados: {det_files}')
print(f'Modelos de reconocimiento encontrados: {rec_files}')

In [ ]:
if det_files:
    dest = f"{DIRS['detection']}/scrfd_10g_kps.onnx"
    shutil.copy2(det_files[0], dest)
    print(f'[OK] SCRFD copiado -> {dest}')
else:
    print('[WARN] No se encontro det_*.onnx, intentando descarga alternativa...')
    # Descarga alternativa directa desde el CDN de insightface
    SCRFD_DIRECT = (
        'https://huggingface.co/deepinsight/scrfd/resolve/main/'
        'SCRFD_10G_KPS.onnx'
    )
    dest = f"{DIRS['detection']}/scrfd_10g_kps.onnx"
    download_file(SCRFD_DIRECT, dest, 'SCRFD_10G_KPS.onnx (directo)'

In [ ]:
#  MODELO 3: ArcFace buffalo_sc Face Recognition ONNX
# Just bundle buffalo_sc contiene w600k_mbf.onnx (MobileFaceNet ArcFace).
# Embeddings: 512 dims, trained in WebFace600K.

In [ ]:
if rec_files:
    dest = f"{DIRS['recognition']}/arcface_mobilefacenet_512d.onnx"
    shutil.copy2(rec_files[0], dest)
    print(f'[OK] ArcFace MobileFaceNet copiado -> {dest}')
else:
    print('[WARN] No se encontro w600k_*.onnx en el bundle.')
    # Descarga alternativa
    ARCFACE_DIRECT = (
        'https://huggingface.co/deepinsight/insightface/resolve/main/'
        'models/buffalo_sc/w600k_mbf.onnx'
    )
    dest = f"{DIRS['recognition']}/arcface_mobilefacenet_512d.onnx"
    download_file(ARCFACE_DIRECT, dest, 'ArcFace MobileFaceNet (directo)')


In [ ]:
# Co0nfirm SCRFD y ArcFace  as onnxruntime
import onnxruntime as ort
import numpy as np

In [ ]:
def verify_onnx_model(model_path: str, input_shape: tuple, model_name: str) -> None:
    print(f'\nVerificando {model_name} ...')
    session = ort.InferenceSession(
        model_path,
        providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
    )
    inputs  = session.get_inputs()
    outputs = session.get_outputs()

    print(f'  Inputs:')
    for inp in inputs:
        print(f'    {inp.name}: {inp.shape} ({inp.type})')
    print(f'  Outputs:')
    for out in outputs:
        print(f'    {out.name}: {out.shape} ({out.type})')

    dummy = np.random.rand(*input_shape).astype(np.float32)
    feed  = {inputs[0].name: dummy}
    result = session.run(None, feed)
    print(f'  Inferencia dummy OK. Salidas: {len(result)} tensores.')
    print(f'[OK] {model_name} verificado.')

In [ ]:
scrfd_path   = f"{DIRS['detection']}/scrfd_10g_kps.onnx"
arcface_path = f"{DIRS['recognition']}/arcface_mobilefacenet_512d.onnx"

verify_onnx_model(scrfd_path,   (1, 3, 640, 640), 'SCRFD-10G-KPS')
verify_onnx_model(arcface_path, (1, 3, 112, 112), 'ArcFace MobileFaceNet')

In [ ]:
#  MiniFASNetV2 Liveness Anti-Spoofing
#  Download weights as PyTorch + Export a ONNX
import torch
import torch.nn as nn

In [ ]:
# Repo: yakhyo/face-anti-spoofing
# Modelo: MiniFASNetV2 
# Input:  [1, 3, 80, 80]  - escala principal
# Output: [1, 2]          - [real_prob, fake_prob]  ## it has a known iissue, they are three porbs not two

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.1.0',
    'torchvision==0.16.0',
], check=True)


In [ ]:
# Clone Repo
if not os.path.exists('/content/face-anti-spoofing'):
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/yakhyo/face-anti-spoofing.git',
        '/content/face-anti-spoofing'
    ], check=True)
    print('[OK] Repo face-anti-spoofing clonado.')

sys.path.insert(0, '/content/face-anti-spoofing')

In [ ]:
# Download Pre-Trained weights MiniFASNetV2
# Repository provide weights at releases/assets
FAS_WEIGHTS_URL = (
    'https://github.com/yakhyo/face-anti-spoofing/releases/download/'
    'v0.0.1/MiniFASNetV2.pth'
)
FAS_WEIGHTS_PATH = '/content/MiniFASNetV2.pth'

download_file(FAS_WEIGHTS_URL, FAS_WEIGHTS_PATH, 'MiniFASNetV2.pth')

In [ ]:
# Load Weights from model
try:
    from src.models.fasnet import MiniFASNetV2
    print('[OK] MiniFASNetV2 importado desde repo.')
except ImportError:
    # Fallback: definicion minima de la arquitectura
    # basada en el paper "Searching Central Difference Convolutional Networks"
    print('[WARN] Import fallo, usando definicion local de MiniFASNetV2.')

    class DepthwiseSeparableConv(nn.Module):
        def __init__(self, in_channels, out_channels, stride=1):
            super().__init__()
            self.dw = nn.Conv2d(
                in_channels, in_channels, 3,
                stride=stride, padding=1, groups=in_channels, bias=False
            )
            self.pw = nn.Conv2d(in_channels, out_channels, 1, bias=False)
            self.bn = nn.BatchNorm2d(out_channels)
            self.relu = nn.ReLU(inplace=True)

        def forward(self, x):
            return self.relu(self.bn(self.pw(self.dw(x))))

    class MiniFASNetV2(nn.Module):
        """
        MiniFASNetV2 - arquitectura ligera para face anti-spoofing.
        Referencia: https://arxiv.org/abs/2011.09671
        """
        def __init__(self, num_classes: int = 2, embedding_size: int = 128):
            super().__init__()
            self.conv1  = nn.Sequential(
                nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True)
            )
            self.block1 = DepthwiseSeparableConv(32,  64,  stride=2)
            self.block2 = DepthwiseSeparableConv(64,  128, stride=2)
            self.block3 = DepthwiseSeparableConv(128, 128, stride=1)
            self.block4 = DepthwiseSeparableConv(128, 256, stride=2)
            self.block5 = DepthwiseSeparableConv(256, 256, stride=1)
            self.block6 = DepthwiseSeparableConv(256, 512, stride=2)
            self.gap    = nn.AdaptiveAvgPool2d(1)
            self.fc     = nn.Linear(512, num_classes)

        def forward(self, x):
            x = self.conv1(x)
            x = self.block1(x)
            x = self.block2(x)
            x = self.block3(x)
            x = self.block4(x)
            x = self.block5(x)
            x = self.block6(x)
            x = self.gap(x)
            x = x.view(x.size(0), -1)
            return self.fc(x)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
model_fas = MiniFASNetV2(num_classes=2)

state_dict = torch.load(FAS_WEIGHTS_PATH, map_location=device

In [ ]:
if isinstance(state_dict, dict) and 'state_dict' in state_dict:
    state_dict = state_dict['state_dict']
elif isinstance(state_dict, dict) and 'model' in state_dict:
    state_dict = state_dict['model']

In [ ]:
#c clean prefix and models 
state_dict = {
    k.replace('module.', ''): v
    for k, v in state_dict.items()
}


In [ ]:
model_fas.load_state_dict(state_dict, strict=False)
model_fas.eval()
model_fas.to(device)

In [ ]:
# Confirm PyTorch Inference 
with torch.no_grad():
    dummy_input = torch.randn(1, 3, 80, 80).to(device)
    output = model_fas(dummy_input)
    probs  = torch.softmax(output, dim=1)
    print(f'Inferencia PyTorch OK. Output shape: {output.shape}')
    print(f'Probabilidades (dummy): real={probs[0][0]:.4f}, fake={probs[0][1]:.4f}')


In [ ]:
# Export MiniFASNetV2 a ONNX
FAS_ONNX_PATH = f"{DIRS['antispoofing']}/minifasnetv2_antispoofing.onnx"

dummy_input_cpu = torch.randn(1, 3, 80, 80)
model_fas_cpu   = model_fas.cpu()

In [ ]:
torch.onnx.export(
    model_fas_cpu,
    dummy_input_cpu,
    FAS_ONNX_PATH,
    export_params    = True,
    opset_version    = 11,
    do_constant_folding = True,
    input_names      = ['input_face'],
    output_names     = ['liveness_logits'],
    dynamic_axes     = {
        'input_face'      : {0: 'batch_size'},
        'liveness_logits' : {0: 'batch_size'},
    }
)
print(f'[OK] MiniFASNetV2 exported to  ONNX -> {FAS_ONNX_PATH}')


In [ ]:
#  Reduce ONNX  Graph
try:
    import onnx
    from onnxsim import simplify

    model_onnx = onnx.load(FAS_ONNX_PATH)
    model_simplified, check = simplify(model_onnx)

    if check:
        onnx.save(model_simplified, FAS_ONNX_PATH)
        print('[OK] Grafo ONNX simplificado con onnxsim.')
    else:
        print('[WARN] onnxsim no pudo simplificar el grafo, se conserva original.')
except Exception as e:
    print(f'[WARN] onnxsim failed (no critico): {e}')

In [ ]:
#  Validate with onnxruntime
verify_onnx_model(FAS_ONNX_PATH, (1, 3, 80, 80), 'MiniFASNetV2 Anti-Spoofing')

In [ ]:
# MODEL IV: EmotiEffLib Emotion Recognition
#           Download wieghts + Export a ONNX

In [ ]:
# Repo: sb-ai-lab/EmotiEffLib
# Backbone: EfficientNet-B0
# Input:  [1, 3, 224, 224]
# Output: [1, 8]  -> 8 emociones
# Clases: anger, contempt, disgust, fear, happiness, neutral, sadness, surprise

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'emotiefflib',
    'timm>=0.9.0',
    'huggingface_hub',
], check=True)

print('[OK] EmotiEffLib y dependencias instaladas.')


In [ ]:
# First Attempt to donwload via official repo packages
try:
    from emotiefflib.facial_analysis import EmotiEffLib

    analyzer = EmotiEffLib(
        model_name = 'enet_b0_8_best_afew',
        engine     = 'torch',
        device     = 'cuda' if torch.cuda.is_available() else 'cpu'
    )
    print('[OK] EmotiEffLib cargado via paquete oficial.')

    # Extraer el modelo PyTorch interno para exportar
    emotion_model = analyzer.model
    emotion_model.eval()

    EMOTION_LABELS = [
        'anger', 'contempt', 'disgust', 'fear',
        'happiness', 'neutral', 'sadness', 'surprise'
    ]
    print(f'Clases de emociones: {EMOTION_LABELS}')

except Exception as e:
    print(f'[WARN] Carga via paquete fallo: {e}')
    print('Intentando descarga directa desde HuggingFace...')

    from huggingface_hub import hf_hub_download
    import timm

    # Descargar pesos desde HuggingFace
    weights_path = hf_hub_download(
        repo_id   = 'sb-ai-lab/EmotiEffLib',
        filename  = 'enet_b0_8_best_afew.pt',
        local_dir = '/content/emotiefflib_weights'
    )
    print(f'[OK] Pesos descargados: {weights_path}')

    # Cargar EfficientNet-B0 con timm y aplicar pesos
    emotion_model = timm.create_model(
        'efficientnet_b0',
        pretrained  = False,
        num_classes = 8
    )
    state = torch.load(weights_path, map_location='cpu')

    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    elif isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']

    state = {k.replace('module.', ''): v for k, v in state.items()}
    emotion_model.load_state_dict(state, strict=False)
    emotion_model.eval()
    print('[OK] EfficientNet-B0 EmotiEffLib cargado con pesos pre-entrenados.')

    EMOTION_LABELS = [
        'anger', 'contempt', 'disgust', 'fear',
        'happiness', 'neutral', 'sadness', 'surprise'
    ]

In [ ]:
# Validate with real time inference  PyTorch
with torch.no_grad():
    dummy_emotion = torch.randn(1, 3, 224, 224)
    out_emotion   = emotion_model(dummy_emotion)
    probs_emotion = torch.softmax(out_emotion, dim=1)
    pred_idx      = torch.argmax(probs_emotion, dim=1).item()
    print(f'Inferencia PyTorch OK. Output shape: {out_emotion.shape}')
    print(f'Prediccion dummy: {EMOTION_LABELS[pred_idx]} ({probs_emotion[0][pred_idx]:.4f})')



In [ ]:
# - Export EmotiEffLib to ONNX

In [ ]:
EMOTION_ONNX_PATH = f"{DIRS['emotion']}/emotiefflib_enet_b0_8emotions.onnx"

dummy_emotion_input = torch.randn(1, 3, 224, 224)
emotion_model_cpu   = emotion_model.cpu()

In [ ]:
torch.onnx.export(
    emotion_model_cpu,
    dummy_emotion_input,
    EMOTION_ONNX_PATH,
    export_params       = True,
    opset_version       = 11,
    do_constant_folding = True,
    input_names         = ['face_crop'],
    output_names        = ['emotion_logits'],
    dynamic_axes        = {
        'face_crop'       : {0: 'batch_size'},
        'emotion_logits'  : {0: 'batch_size'},
    }
)
print(f'[OK] EmotiEffLib exportado a ONNX -> {EMOTION_ONNX_PATH}')

In [ ]:
# Reduce Graph
try:
    import onnx
    from onnxsim import simplify

    model_onnx = onnx.load(EMOTION_ONNX_PATH)
    model_simplified, check = simplify(model_onnx)

    if check:
        onnx.save(model_simplified, EMOTION_ONNX_PATH)
        print('[OK] Grafo ONNX simplificado.')
except Exception as e:
    print(f'[WARN] onnxsim fallo (no critico): {e}')

In [ ]:
# Validate with onnxruntime
verify_onnx_model(EMOTION_ONNX_PATH, (1, 3, 224, 224), 'EmotiEffLib EfficientNet-B0')

In [ ]:
# SAVE MODELS METADATA
import json
from datetime import datetime

In [ ]:
metadata = {
    'export_date'  : datetime.now().isoformat(),
    'colab_runtime': 'T4 GPU',
    'models': {
        'face_detection': {
            'name'         : 'SCRFD-10G-KPS',
            'file'         : 'scrfd_10g_kps.onnx',
            'source'       : 'deepinsight/insightface',
            'input_shape'  : [1, 3, 640, 640],
            'input_name'   : 'input.1',
            'output'       : 'bounding_boxes + 5_keypoints',
            'preprocessing': {
                'resize'   : [640, 640],
                'normalize': 'mean=[127.5,127.5,127.5] std=[128,128,128]',
                'format'   : 'BGR'
            },
            'notes': 'Entrega bbox [x1,y1,x2,y2] + 5 keypoints para alineacion afin'
        },
        'liveness_antispoofing': {
            'name'         : 'MiniFASNetV2',
            'file'         : 'minifasnetv2_antispoofing.onnx',
            'source'       : 'yakhyo/face-anti-spoofing',
            'input_shape'  : [1, 3, 80, 80],
            'input_name'   : 'input_face',
            'output_name'  : 'liveness_logits',
            'output_shape' : [1, 2],
            'classes'      : ['real', 'fake'],
            'threshold'    : 0.6,
            'preprocessing': {
                'resize'   : [80, 80],
                'normalize': 'mean=[0.485,0.456,0.406] std=[0.229,0.224,0.225]',
                'format'   : 'RGB'
            },
            'notes': 'softmax(output)[0] = prob_real. Rechazar si prob_real < 0.6'
        },
        'face_recognition': {
            'name'          : 'ArcFace MobileFaceNet (buffalo_sc)',
            'file'          : 'arcface_mobilefacenet_512d.onnx',
            'source'        : 'deepinsight/insightface',
            'input_shape'   : [1, 3, 112, 112],
            'output_shape'  : [1, 512],
            'embedding_dims': 512,
            'preprocessing' : {
                'resize'    : [112, 112],
                'normalize' : 'mean=[0.5,0.5,0.5] std=[0.5,0.5,0.5]',
                'format'    : 'RGB',
                'align'     : 'affine_transform con 5 keypoints de SCRFD'
            },
            'similarity'    : 'cosine',
            'threshold'     : 0.6,
            'notes'         : 'Normalizar embedding con L2 antes de almacenar en pgvector'
        },
        'emotion_recognition': {
            'name'         : 'EmotiEffLib EfficientNet-B0',
            'file'         : 'emotiefflib_enet_b0_8emotions.onnx',
            'source'       : 'sb-ai-lab/EmotiEffLib',
            'model_variant': 'enet_b0_8_best_afew',
            'input_shape'  : [1, 3, 224, 224],
            'input_name'   : 'face_crop',
            'output_name'  : 'emotion_logits',
            'output_shape' : [1, 8],
            'classes'      : [
                'anger', 'contempt', 'disgust', 'fear',
                'happiness', 'neutral', 'sadness', 'surprise'
            ],
            'preprocessing': {
                'resize'   : [224, 224],
                'normalize': 'mean=[0.485,0.456,0.406] std=[0.229,0.224,0.225]',
                'format'   : 'RGB'
            },
            'notes': 'softmax(output) para obtener probabilidades por clase'
        }
    },
    'pipeline_flow': [
        '1. SCRFD: frame 640x640 -> bbox + 5 keypoints',
        '2. affine_align: crop alineado 80x80 y 112x112 y 224x224',
        '3. MiniFASNetV2: 80x80 -> [real, fake] probs',
        '4. ArcFace: 112x112 -> embedding[512] -> cosine vs pgvector',
        '5. EmotiEffLib: 224x224 -> emotion_logits[8] (paralelo con paso 4)'
    ]
}

In [ ]:
META_PATH = f'{DRIVE_BASE}/models_metadata.json'
with open(META_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f'[OK] Metadata guardada -> {META_PATH}')

In [ ]:
# Confirm Models are SAVE in MyDrive

In [ ]:
all_models = [
    (scrfd_path,                                'SCRFD-10G-KPS          [Face Detection]'),
    (FAS_ONNX_PATH,                             'MiniFASNetV2           [Anti-Spoofing]'),
    (arcface_path,                              'ArcFace MobileFaceNet  [Face Recognition]'),
    (EMOTION_ONNX_PATH,                         'EmotiEffLib EfficientNet-B0 [Emotion]'),
]

In [ ]:
total_mb = 0.0
for model_path, label in all_models:
    if os.path.exists(model_path):
        size_mb   = os.path.getsize(model_path) / (1024 * 1024)
        total_mb += size_mb
        print(f'  [OK] {label}')
        print(f'       Path: {model_path}')
        print(f'       Size: {size_mb:.1f} MB')
    else:
        print(f'  [FAIL] {label} - archivo no encontrado en {model_path}')

In [ ]:
print(f'\nTotal modelos: {total_mb:.1f} MB')
print(f'Directorio raiz en Drive: {DRIVE_BASE}')
print(f'  {DRIVE_BASE}/')
print(f'  |-- 01_face_detection/scrfd_10g_kps.onnx')
print(f'  |-- 02_liveness_antispoofing/minifasnetv2_antispoofing.onnx')
print(f'  |-- 03_face_recognition/arcface_mobilefacenet_512d.onnx')
print(f'  |-- 04_emotion_recognition/emotiefflib_enet_b0_8emotions.onnx')
print(f'  |-- models_metadata.json')

In [ ]:
# Download to Local Machine 

# from google.colab import files
#
# for model_path, label in all_models:
#     if os.path.exists(model_path):
#         print(f'Descargando {label} ...')
#         files.download(model_path)
#
# files.download(META_PATH)# FACE PIPELINE - ONNX EXPORT NOTEBOOK